In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import copy
import gc
import glob
import math
import os
import pathlib
import time
from collections import OrderedDict
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import xarray as xr
from tqdm.notebook import tqdm

plt.rcParams["font.family"] = "serif"
plt.style.use("tableau-colorblind10")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = r":4096:8"  # to make calculations deterministic

In [ ]:
from scripts.train_pdd_lorenz96 import initialize_trainer
from src.configs.lorenz96_config import Lorenz96UnetConfig
from src.util.random_seed_helper import set_seeds

# Define constants

In [ ]:
DEVICE = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
ROOT_DIR = pathlib.Path(os.environ["PYTHONPATH"].split(":")[0]).resolve()
print(f"{ROOT_DIR=}, {DEVICE=}")

# Plot deep-learning data

In [ ]:
path = f"{ROOT_DIR}/data/DL_data/lorenz96/lorenz96_K32J04_b10p0_c10p0_F10p0_sigma0p0.nc"
da = xr.load_dataarray(path)

In [ ]:
data = da.sel(batch=0).values
assert data.shape == (2, 64, 128)  # channel, time, space

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ax in enumerate(axes):
    ax.set_xticks([])
    ax.set_yticks([])
    d = data[i].transpose()
    xs = np.arange(d.shape[0])
    ts = np.arange(d.shape[1])
    xs, ts = np.meshgrid(xs, ts, indexing="ij")
    cmap = "BrBG_r" if i == 0 else "twilight_shifted"
    ret = ax.pcolormesh(xs, ts, d, shading="nearest", cmap=cmap)
    fig.colorbar(ret, ax=ax)

# Make config

In [ ]:
config = Lorenz96UnetConfig(
    laplacian_factor=0.1,
    mass_factor=0.0,
    window_size=5,
    noise_amplitude_squared=2.0,
    seed=771155,
)
config.save(f"{ROOT_DIR}/configs/lorenz96_unet.yml")

# Load a trained PDD

In [ ]:
p = f"{ROOT_DIR}/configs/lorenz96_unet.yml"
config = Lorenz96UnetConfig.load(p)
p = f"{ROOT_DIR}/data/DL_model/lorenz96/lorenz96_unet"
trainer, dataset = initialize_trainer(
    config, str(DEVICE), ROOT_DIR, result_dir=p, kind="test"
)
trainer.load_only_model(milestone=30_000)
_ = trainer.model.noise_estimate_fn.closure.eval()

# Perform simulation

# Perform generation

In [ ]:
set_seeds(42)
intermediates = trainer.model.sample(
    batch_size=5,
    corrector_snr=0.7,
    num_corrector_steps=3,
)

In [ ]:
data = dataset.standardize_inversely(intermediates[1][0].numpy())
assert data.shape == (2, 64, 128)  # channel, time, space

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ax in enumerate(axes):
    ax.set_xticks([])
    ax.set_yticks([])
    d = data[i].transpose()
    xs = np.arange(d.shape[0])
    ts = np.arange(d.shape[1])
    xs, ts = np.meshgrid(xs, ts, indexing="ij")
    cmap = "BrBG_r" if i == 0 else "twilight_shifted"
    ret = ax.pcolormesh(xs, ts, d, shading="nearest", cmap=cmap)
    fig.colorbar(ret, ax=ax)